# Phase 2: The Baseline Model
This notebook implements the baseline architecture: XLM-R Embedding -> 1D CNN -> LSTM -> Classifier.

In [ ]:
import torch
import torch.nn as nn
from transformers import XLMRobertaModel

class BaselineFakeNewsModel(nn.Module):
    def __init__(self, model_name='xlm-roberta-base'):
        super(BaselineFakeNewsModel, self).__init__()
        # 1. The Embedding Engine
        self.xlm_roberta = XLMRobertaModel.from_pretrained(model_name)
        
        # 2. The Pattern Catcher (Conv1D)
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=768, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 3. The Context Builder (LSTM)
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, batch_first=True)
        
        # 4. The Classifier
        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        cnn_input = sequence_output.transpose(1, 2)
        cnn_output = self.cnn(cnn_input)
        
        lstm_input = cnn_output.transpose(1, 2)
        _, (hn, _) = self.lstm(lstm_input)
        
        logits = self.fc(hn.squeeze(0))
        return self.sigmoid(logits)

print("Baseline Model Architecture defined.")

## Training and Evaluation
The model is trained on English and Hindi data and evaluated using the F1-Score.